In [ ]:
# RQ5 (stretch, derived): the cost to re-establish verifiable state after record updates, assuming each update
# triggers the implementation's full rebuild (spec 2, H5). No new runs: every number is a Tier 1 grid median.
import json, sys
from pathlib import Path

import pandas as pd

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.analysis import amortise, frame, plots
from cs4406_sem_project.harness import loadgen, session_check
from cs4406_sem_project.harness.grid import fmt_size

RUNS = [repo / "results/raw/all-20261112_tier1", repo / "results/raw/all-20261112_anchors"]  # notebook 17
platform = json.loads((RUNS[0] / "meta.json").read_text())["platform"]
df, status = frame.load_runs(RUNS, platform)
FIG = repo / "docs/figures"
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 400)


def save(name, fig):
    for ext in ("pdf", "png"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=200, bbox_inches="tight")


plots._style()
CELL = (128 << 20, 1 << 10)  # decision 28: the RQ4 cell, 128 MiB x 1 KiB (S3 at 1 bit)

t = plots.build(df, status)
med = t.medians
print(f"{len(med)} cells with medians")

In [ ]:
# Rebuild cost per update = one whole offline phase: server seconds (preprocessing, plus the proof where there is
# one), client seconds (offline verification) and offline bytes (the hint or digest the client downloads again).
# M0 keeps no offline state. 1 KiB records (S3 per bit) at every size each arm ran.
cols = ["server_offline_s", "client_offline_s", "offline_bytes", "server_online_s", "client_online_s", "online_bytes"]
reb = med[(med.record_bytes == 1 << 10) | (med.arm == "S3")][["arm", "db_bytes", "record_bytes", *cols]].copy()
reb["db"] = reb.db_bytes.map(lambda v: fmt_size(int(v)))
print(reb.sort_values(["db_bytes", "arm"])[["arm", "db", *cols]].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

In [ ]:
# H5: per query, at u updates per query (each forcing a rebuild before the next query), the cost is
# online + u x offline, which is amortise's cost at Q = 1 / u. Rank the arms at 128 MiB and 1 GiB x 1 KiB.
import math
US = (0, 0.01, 0.1, 1)
for db in (128 << 20, 1 << 30):
    sel = med[(med.db_bytes == db) & (med.record_bytes == 1 << 10)].set_index("arm")
    for kind in ("total_s", "bytes"):
        fn = amortise.AMORTISED[kind]
        out = pd.DataFrame({f"u={u}": fn(sel, math.inf if u == 0 else 1 / u) for u in US})
        print(f"{fmt_size(db)} x 1 KiB, {kind} per query\n{out.to_string(float_format=lambda v: f'{v:.4g}')}")
        print("most expensive:", {c: out[c].idxmax() for c in out}, "cheapest:", {c: out[c].idxmin() for c in out}, "\n")

In [ ]:
# The update rate at which rebuilding dominates: u* = online / offline, where the rebuild share of a query's cost
# reaches one half, per arm at each size (client + server seconds). A lower u* means updates hurt sooner.
reb["u_half_time"] = (reb.server_online_s + reb.client_online_s.fillna(0)) / (reb.server_offline_s + reb.client_offline_s.fillna(0))
print(reb.pivot_table(index="arm", columns="db", values="u_half_time").to_string(float_format=lambda v: f"{v:.3g}"))